In [25]:
import pandas as pd

# ============================================
# Step 1: Load the original dataset
# ============================================
df = pd.read_csv('../data/raw/Movies.csv')
df.head()

,MovieID,Title,Genres,Language,Country,Total Views
0,1,Toy Story (1995),Animation|Children's|Comedy,German,Italy,69118
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Spanish,UK,1661
2,3,Grumpier Old Men (1995),Comedy|Romance,English,India,22205
3,4,Waiting to Exhale (1995),Comedy|Drama,Spanish,India,84783
4,5,Father of the Bride Part II (1995),Comedy,German,India,61291


In [27]:
# ============================================
# Step 2: Clean titles and extract year
# ============================================
df[['Movie_Title', 'Year']] = df['Title'].str.extract(r'(.*?)\s*\((\d{4})\)$')
df['Movie_Title'] = df['Movie_Title'].fillna('Unknown')
df['Year'] = df['Year'].fillna('Unknown')

In [30]:
# ============================================
# Step 3: Fix known data quality issues
# ============================================
df.loc[df['Movie_Title'] == 'Final Destination', 'Genres'] = 'Horror'
df.loc[df['Movie_Title'] == "Schindler's List", 'Genres'] = 'War|Drama'
df.loc[df['Movie_Title'] == 'Gladiator', 'Genres'] = 'Action|Drama'
df.loc[df['Movie_Title'] == 'Scary Movie', 'Genres'] = 'Comedy|Horror'

df['Genres'] = df['Genres'].str.replace('Dramma', 'Drama', regex=False)
df['Genres'] = df['Genres'].str.replace('Dramatic', 'Drama', regex=False)

# Show the updated rows
print(df[df['MovieID'].isin([16, 3936])][['MovieID', 'Title', 'Movie_Title', 'Genres']].to_string(index=False))

print(df[df['Movie_Title'].isin(['Final Destination', "Schindler's List", 'Gladiator', 'Scary Movie'])][['MovieID', 'Title', 'Movie_Title', 'Genres']].to_string(index=False))

 MovieID            Title Movie_Title         Genres
      16 Casino (not_def)     Unknown Drama|Thriller
    3936            -1943     Unknown Drama|Thriller
 MovieID                    Title       Movie_Title        Genres
     527  Schindler's List (1993)  Schindler's List     War|Drama
    3409 Final Destination (2000) Final Destination        Horror
    3578         Gladiator (2000)         Gladiator  Action|Drama
    3785       Scary Movie (2000)       Scary Movie Comedy|Horror


In [31]:
# ============================================
# Step 4: Create GENRES master table
# ============================================
all_genres = (
    df['Genres']
    .str.split('|')
    .explode()
    .dropna()
    .unique()
)

genres_df = pd.DataFrame({'Genre_Name': all_genres})
genres_df = genres_df.sort_values('Genre_Name').reset_index(drop=True)
genres_df.insert(0, 'GenreID', range(1, 1 + len(genres_df)))

genres_df.head()

,GenreID,Genre_Name
0,1,Action
1,2,Adventure
2,3,Animation
3,4,Children's
4,5,Comedy


In [32]:
# ============================================
# Step 5: Create MOVIE_GENRES junction table
# ============================================
junction_df = df[['MovieID', 'Genres']].copy()
junction_df['Genres'] = junction_df['Genres'].str.split('|')
junction_df = junction_df.explode('Genres').rename(columns={'Genres': 'Genre_Name'})

genre_to_id_map = dict(zip(genres_df['Genre_Name'], genres_df['GenreID']))
junction_df['GenreID'] = junction_df['Genre_Name'].map(genre_to_id_map)

movie_genres_junction = junction_df[['MovieID', 'GenreID']].dropna().astype(int)

movie_genres_junction.head()

,MovieID,GenreID
0,1,3
0,1,4
0,1,5
1,2,2
1,2,4


In [34]:
# ============================================
# Step 6: Create cleaned MOVIES table
# ============================================
movies_clean_df = df[['MovieID', 'Movie_Title', 'Year', 'Language', 'Country', 'Total Views']].copy()

movies_clean_df.head()

,MovieID,Movie_Title,Year,Language,Country,Total Views
0,1,Toy Story,1995,German,Italy,69118
1,2,Jumanji,1995,Spanish,UK,1661
2,3,Grumpier Old Men,1995,English,India,22205
3,4,Waiting to Exhale,1995,Spanish,India,84783
4,5,Father of the Bride Part II,1995,German,India,61291


In [35]:
# ============================================
# Step 7: Save output files
# ============================================
genres_df.to_csv('../data/processed/genres.csv', index=False)
movie_genres_junction.to_csv('../data/processed/movie_genres.csv', index=False)
movies_clean_df.to_csv('../data/processed/movies_clean.csv', index=False)

print("Normalization complete! 3 CSV files generated successfully.")

Normalization complete! 3 CSV files generated successfully.
